# HBCC cũ — huấn luyện với augmentation công bằng

Notebook này là pipeline chính cho **HBCC-Small và HBCC-Medium hiện tại**. Sáu mô hình trong bảng report (ResNet-18, MobileNetV2, ShuffleNetV2, CoC, HBCC-Small và HBCC-Medium) dùng đúng một recipe, một split và cùng seed 17. P-HBCC-2M không thuộc ma trận mặc định.

Recipe có RandomCrop, horizontal flip, Random Erasing, MixUp/CutMix và label smoothing; RandAugment được tắt. Mặc định chỉ chạy smoke test. Đặt `RUN_FULL=True` khi sẵn sàng chạy 6 mô hình × 1 seed × 300 epochs = 6 runs.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

REPO_ROOT = None  # Có thể gán thủ công trên Kaggle hoặc Windows.

cwd = Path.cwd().resolve()
candidates = [Path(REPO_ROOT).resolve()] if REPO_ROOT else [cwd, *cwd.parents]
for base in (Path('/kaggle/working'), Path('/kaggle/input')):
    if base.exists():
        candidates.extend(path.parent.parent for path in base.glob('*/lightweight_hbcc/__init__.py'))
        candidates.extend(path.parent.parent for path in base.glob('*/*/lightweight_hbcc/__init__.py'))
ROOT = next((path for path in candidates if (path / 'lightweight_hbcc').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Không tìm thấy thư mục gốc Lightweight-Context-Cluster; hãy gán REPO_ROOT.')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PYTHON = sys.executable
print('Repository:', ROOT)
print('Python    :', PYTHON)

## 1. Cấu hình

Trên Kaggle, `DATA_ROOT` phải là thư mục cha chứa trực tiếp `cifar-100-python/` hoặc `cifar-10-batches-py/`. Giữ `PREPARE_DATA=False` nếu đã gắn dataset qua Kaggle Input; chỉ bật khi cần tải một lần và Internet hoạt động.

In [ ]:
from tools.run_fair_comparison import (
    BASELINE_MODELS,
    CORE_MODELS as RUNNER_CORE_MODELS,
    DEFAULT_SEEDS as RUNNER_DEFAULT_SEEDS,
    HBCC_MODELS,
)

DATASET = 'cifar100'  # 'cifar10' hoặc 'cifar100'
KAGGLE_DATA_ROOT = None  # Ví dụ: '/kaggle/input/cifar100-python'
root_posix = ROOT.as_posix()
WRITABLE_ROOT = Path('/kaggle/working') if root_posix.startswith('/kaggle/input/') else ROOT
DATA_ROOT = Path(KAGGLE_DATA_ROOT) if KAGGLE_DATA_ROOT else WRITABLE_ROOT / 'data'
PREPARE_DATA = False

FULL_MODELS = list(RUNNER_CORE_MODELS)
FULL_SEEDS = list(RUNNER_DEFAULT_SEEDS)
SMOKE_MODELS = ['hbcc_small']
SMOKE_SEEDS = [FULL_SEEDS[0]]

RUN_SMOKE = True
FORCE_SMOKE = False
RUN_FULL = False
RUN_BENCHMARK_AFTER_FULL = True

OUTPUT_ROOT = WRITABLE_ROOT / 'runs_fair_paper_inspired_300e'
BENCHMARK_ROOT = WRITABLE_ROOT / 'results' / 'fair_paper_inspired_300e'
REPORT_ROOT = WRITABLE_ROOT / 'results' / 'hbcc_old_fair_report_300e' / DATASET

assert DATASET in {'cifar10', 'cifar100'}
assert set(HBCC_MODELS) == {'hbcc_small', 'hbcc_medium'}
assert 'phbcc_2m' not in FULL_MODELS
print('Dataset    :', DATASET)
print('Writable   :', WRITABLE_ROOT)
print('Data root  :', DATA_ROOT)
print('Baselines  :', list(BASELINE_MODELS))
print('HBCC models:', list(HBCC_MODELS))
print('Seeds      :', FULL_SEEDS)
print('Total runs :', len(FULL_MODELS) * len(FULL_SEEDS))

## 2. Chuẩn bị dữ liệu một lần

Cell này tách download khỏi bước train, tránh tình trạng Kaggle đứng lâu tại `building data loaders...`. Khi `PREPARE_DATA=False`, dữ liệu thiếu sẽ báo lỗi ngay thay vì âm thầm chờ mạng.

In [ ]:
import time
from torchvision.datasets import CIFAR10, CIFAR100

dataset_class = CIFAR100 if DATASET == 'cifar100' else CIFAR10
DATA_ROOT.mkdir(parents=True, exist_ok=True)
start = time.perf_counter()
try:
    train_data = dataset_class(DATA_ROOT, train=True, download=PREPARE_DATA)
    test_data = dataset_class(DATA_ROOT, train=False, download=PREPARE_DATA)
except RuntimeError as exc:
    raise RuntimeError(
        f'{DATASET} chưa tồn tại đúng cấu trúc tại {DATA_ROOT}. '
        'Hãy đặt DATA_ROOT tới Kaggle Input đã giải nén hoặc bật PREPARE_DATA=True một lần.'
    ) from exc
print(f'Data ready: train={len(train_data)} test={len(test_data)} elapsed={time.perf_counter()-start:.2f}s')
del train_data, test_data

## 3. Preflight augmentation fairness

Runner sẽ từ chối chạy nếu bất kỳ mô hình nào lệch toàn bộ khối `data`, `train` hoặc `protocol`. Cell cũng hiển thị transform thực tế để lưu làm bằng chứng trong report.

In [ ]:
import pandas as pd

from lightweight_hbcc import data as data_module
from tools.run_fair_comparison import config_paths, validate_controlled_configs

fair_configs = validate_controlled_configs(DATASET, config_paths(DATASET, FULL_MODELS))
reference_cfg = fair_configs[FULL_MODELS[0]]
reference_contract = {key: reference_cfg[key] for key in ('data', 'train', 'protocol')}
for model_name, cfg in fair_configs.items():
    assert {key: cfg[key] for key in ('data', 'train', 'protocol')} == reference_contract, model_name

train_transform = data_module._transforms(DATASET, True, True, reference_cfg['data'])
eval_transform = data_module._transforms(DATASET, False, False, reference_cfg['data'])
recipe_rows = [
    ('protocol', reference_cfg['protocol']['name']),
    ('epochs', reference_cfg['train']['epochs']),
    ('split_seed', reference_cfg['data']['split_seed']),
    ('batch_size', reference_cfg['data']['batch_size']),
    ('RandAugment enabled', reference_cfg['data']['randaugment']['enabled']),
    ('Random Erasing p', reference_cfg['data']['random_erasing']['p']),
    ('MixUp alpha', reference_cfg['train']['mixup_alpha']),
    ('CutMix alpha', reference_cfg['train']['cutmix_alpha']),
    ('CutMix switch p', reference_cfg['train']['cutmix_prob']),
    ('label smoothing', reference_cfg['train']['label_smoothing']),
    ('train transform', repr(train_transform)),
    ('eval transform', repr(eval_transform)),
]
display(pd.DataFrame(recipe_rows, columns=['fairness item', 'shared value']))
print(f'PASS: {len(fair_configs)} mô hình dùng cùng một augmentation/training contract.')

## 4. Kiểm tra hai kiến trúc HBCC cũ

In [ ]:
import torch
from lightweight_hbcc.models import build_model

architecture_rows = []
expected_classes = 100 if DATASET == 'cifar100' else 10
for model_name in HBCC_MODELS:
    cfg = fair_configs[model_name]
    model = build_model(cfg).eval()
    with torch.inference_mode():
        logits = model(torch.randn(1, 3, 32, 32))
    assert logits.shape == (1, expected_classes)
    architecture_rows.append({
        'model': model_name,
        'embed_dims': cfg['model']['embed_dims'],
        'depths': cfg['model']['depths'],
        'params': sum(parameter.numel() for parameter in model.parameters()),
        'logits': tuple(logits.shape),
    })
    del model, logits
display(pd.DataFrame(architecture_rows))

## 5. Smoke test và full training

Smoke chỉ chạy một batch mỗi split và không có giá trị nghiên cứu. Full run mặc định gồm 6 lượt tuần tự; runner bỏ qua duy nhất các artifact hoàn tất có recipe khớp chính xác.

In [ ]:
def run_command(command):
    command = [str(item) for item in command]
    print(shlex.join(command), flush=True)
    return subprocess.run(command, cwd=ROOT, check=True)

def fair_command(models, seeds, *, smoke=False, benchmark=False):
    command = [
        PYTHON, 'tools/run_fair_comparison.py',
        '--dataset', DATASET,
        '--models', *models,
        '--seeds', *map(str, seeds),
        '--data-root', DATA_ROOT,
        '--output', OUTPUT_ROOT,
        '--benchmark-output', BENCHMARK_ROOT,
    ]
    if smoke:
        command.append('--smoke')
    if benchmark:
        command.append('--benchmark')
    return command

smoke_command = fair_command(SMOKE_MODELS, SMOKE_SEEDS, smoke=True)
if FORCE_SMOKE:
    smoke_command.append('--force')
if RUN_SMOKE:
    run_command(smoke_command)
else:
    print('Smoke command:', shlex.join(map(str, smoke_command)))

In [ ]:
full_command = fair_command(
    FULL_MODELS,
    FULL_SEEDS,
    benchmark=RUN_BENCHMARK_AFTER_FULL,
)
if RUN_FULL:
    run_command(full_command)
else:
    print('Full training chưa chạy. Đặt RUN_FULL=True khi sẵn sàng.')
    print(shlex.join(map(str, full_command)))

## 6. Đọc và khóa coverage seed 17

Chỉ run canonical đúng protocol, đúng 300 epoch, seed 17 và thuộc sáu mô hình chính mới được tổng hợp. Thiếu bất kỳ mô hình nào thì bảng chính bị khóa. Vì chỉ chạy một seed, notebook không tính standard deviation hoặc confidence interval.

In [ ]:
from lightweight_hbcc.config import load_config

rows = []
param_cache = {}
for metrics_path in sorted(OUTPUT_ROOT.glob('*/test_metrics.json')):
    run_dir = metrics_path.parent
    config_path = run_dir / 'config.yaml'
    if not config_path.exists():
        continue
    cfg = load_config(config_path)
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    run_name = run_dir.name
    architecture = run_name.split('_seed', 1)[0].replace(f'fair_{DATASET}_', '')
    if architecture not in FULL_MODELS or cfg.get('data', {}).get('name') != DATASET:
        continue
    if architecture not in param_cache:
        counted_model = build_model(cfg)
        param_cache[architecture] = sum(parameter.numel() for parameter in counted_model.parameters())
        del counted_model
    rows.append({
        'run': run_name,
        'architecture': architecture,
        'seed': int(cfg['train']['seed']),
        'protocol': cfg.get('protocol', {}).get('name'),
        'canonical': bool(cfg.get('protocol', {}).get('canonical', False)),
        'epochs': int(cfg['train']['epochs']),
        'params': param_cache[architecture],
        'test_acc1': metrics.get('test_acc1'),
    })

results = pd.DataFrame(rows)
if results.empty:
    print('Chưa có kết quả canonical trong', OUTPUT_ROOT)
else:
    display(results.sort_values(['architecture', 'seed']))

In [ ]:
summary = pd.DataFrame()
single_seed_differences = pd.DataFrame()

if not results.empty:
    canonical = results[
        results['canonical']
        & results['protocol'].eq(reference_cfg['protocol']['name'])
        & results['epochs'].eq(int(reference_cfg['protocol']['effective_epochs']))
        & results['test_acc1'].notna()
    ].copy()
    duplicate_counts = canonical.groupby(['architecture', 'seed']).size()
    if duplicate_counts.gt(1).any():
        raise RuntimeError(f'Trùng architecture/seed: {duplicate_counts[duplicate_counts.gt(1)].to_dict()}')

    expected_seeds = set(map(int, FULL_SEEDS))
    coverage_rows = []
    for architecture in FULL_MODELS:
        present = set(canonical.loc[canonical['architecture'].eq(architecture), 'seed'].astype(int))
        coverage_rows.append({
            'architecture': architecture,
            'present': sorted(present),
            'missing': sorted(expected_seeds - present),
            'complete': present == expected_seeds,
        })
    coverage = pd.DataFrame(coverage_rows)
    display(coverage)

    if not bool(coverage['complete'].all()):
        print('Chưa đủ seed 17 cho toàn bộ mô hình; không tạo bảng chính.')
    else:
        pivot = canonical.pivot(index='seed', columns='architecture', values='test_acc1').sort_index()
        if pivot[FULL_MODELS].isna().any().any():
            raise RuntimeError('Ma trận model/seed có ô trống.')
        summary = (
            canonical[['architecture', 'seed', 'test_acc1', 'params']]
            .rename(columns={'test_acc1': 'accuracy_seed17'})
            .sort_values('accuracy_seed17', ascending=False)
        )
        summary['params_m'] = summary['params'] / 1e6
        display(summary)

        comparison_rows = []
        for target in HBCC_MODELS:
            for baseline in BASELINE_MODELS:
                delta = float(pivot.loc[FULL_SEEDS[0], target] - pivot.loc[FULL_SEEDS[0], baseline])
                comparison_rows.append({
                    'hbcc_model': target,
                    'baseline': baseline,
                    'seed': FULL_SEEDS[0],
                    'delta_pp': delta,
                    'hbcc_higher': delta > 0,
                })
        single_seed_differences = pd.DataFrame(comparison_rows)
        display(single_seed_differences)

        REPORT_ROOT.mkdir(parents=True, exist_ok=True)
        summary.to_csv(REPORT_ROOT / 'summary.csv', index=False)
        single_seed_differences.to_csv(REPORT_ROOT / 'single_seed_differences.csv', index=False)
        print('Saved report tables to', REPORT_ROOT)

## Quy tắc đưa vào report

1. Chỉ dùng kết quả từ notebook/runner fair này; không trộn kết quả từ `full_training_pipeline.ipynb` hoặc `cifar100_training_pipeline.ipynb` cũ.
2. Chỉ báo cáo run `canonical=true`, đúng protocol, đúng 300 epochs và seed 17.
3. Ghi rõ mọi mô hình dùng cùng augmentation, split seed 42 và train seed 17.
4. Báo accuracy của seed 17, chênh lệch trực tiếp, parameters và latency/MAC. Không báo mean ± SD, confidence interval hay ý nghĩa thống kê từ một seed.
5. P-HBCC-2M không được đưa vào bảng kết luận chính.